# Customer Churn Prediction with Logistic Regression

**Aim:** Predict which telecom customers are likely to churn (cancel their service) from their account and billing data, so a company could contact them with a retention offer *before* they leave.

**How success is measured:** Missing a churner (false negative) costs the company a customer, while an unnecessary retention offer (false positive) costs a small discount or a phone call. Because of that asymmetry, the model is judged mainly on **recall for the churn class**, balanced against precision, rather than on accuracy alone.

## Project objectives
- Clean and encode the raw telecom data for a linear model, including fixing redundant dummy columns.
- Train a logistic regression baseline and check whether feature scaling changes the result.
- Address class imbalance (about 27% of customers churn) with threshold tuning and class weighting.
- Choose a final configuration and justify it with the precision/recall trade-off.

## Dataset
Telco Customer Churn (IBM sample data, available on Kaggle): 7,043 customers, 20 features (demographics, subscribed services, contract and billing details) and the target `Churn` (Yes/No). The notebook reads the file as `telco_cust.csv` from the same folder.

**Label convention:** `Churn = 1` is the positive class. Confusion matrices are laid out as `[[TN, FP], [FN, TP]]`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split

## 1. Data loading and cleaning

Load the data and inspect types and missing values. `TotalCharges` is stored as text in the raw file (blank entries for some customers), so it is converted to numeric. The 11 resulting missing values are filled with the column median.

In [2]:
# Load the dataset

df = pd.read_csv('telco_cust.csv')

In [3]:
print(f'Dataset shape: {df.shape}')
print('\nFirst few rows:')
print(df.head())

Dataset shape: (7043, 21)

First few rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport Strea

In [4]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [5]:
print('\nSummary statistics:')
print(df.describe(include='all'))


Summary statistics:
        customerID gender  SeniorCitizen Partner Dependents       tenure  \
count         7043   7043    7043.000000    7043       7043  7043.000000   
unique        7043      2            NaN       2          2          NaN   
top     7590-VHVEG   Male            NaN      No         No          NaN   
freq             1   3555            NaN    3641       4933          NaN   
mean           NaN    NaN       0.162147     NaN        NaN    32.371149   
std            NaN    NaN       0.368612     NaN        NaN    24.559481   
min            NaN    NaN       0.000000     NaN        NaN     0.000000   
25%            NaN    NaN       0.000000     NaN        NaN     9.000000   
50%            NaN    NaN       0.000000     NaN        NaN    29.000000   
75%            NaN    NaN       0.000000     NaN        NaN    55.000000   
max            NaN    NaN       1.000000     NaN        NaN    72.000000   

       PhoneService MultipleLines InternetService OnlineSecurity  

In [6]:
# Convert the numeric column to a proper numeric type and handle missing values

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(df['TotalCharges'].head(20))
print(f'Missing values in TotalCharges: {df["TotalCharges"].isnull().sum()}')

median_val = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(median_val)
print(f'Remaining missing values: {df["TotalCharges"].isnull().sum()}')

0       29.85
1     1889.50
2      108.15
3     1840.75
4      151.65
5      820.50
6     1949.40
7      301.90
8     3046.05
9     3487.95
10     587.45
11     326.80
12    5681.10
13    5036.30
14    2686.05
15    7895.15
16    1022.95
17    7382.25
18     528.35
19    1862.90
Name: TotalCharges, dtype: float64
Missing values in TotalCharges: 11
Remaining missing values: 0


## 2. Feature engineering and encoding

Logistic regression needs numeric inputs, so each column type is handled differently:

- **Binary Yes/No columns** (`Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, `Churn`) and `gender` are mapped to 0/1.
- **Add-on service columns** (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) and `MultipleLines` contain a third value, "No internet service" / "No phone service". That fact is already captured by `InternetService` and `PhoneService`, so it is collapsed into "No" and the columns become plain 0/1. One-hot encoding them directly created seven identical "no internet service" dummy columns, a perfect multicollinearity problem that the correlation matrix exposed.
- **`Contract`** has a natural order (month-to-month < one year < two year), so it is encoded as an ordinal 0/1/2.
- **`InternetService` and `PaymentMethod`** are nominal with no order, so they are one-hot encoded with `drop_first=True` (DSL and Bank transfer become the implicit baselines).
- **`customerID`** is dropped because it carries no predictive information.

In [7]:
# Encode binary and ordinal labels

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

binary_cols = ['Partner', 'Dependents', 'PaperlessBilling', 'PhoneService']
df[binary_cols] = df[binary_cols].replace({'Yes': 1, 'No': 0}).astype(int)
df['gender'] = df['gender'].map({'Female': 1, 'Male': 0})

In [8]:
# Standardize label variations before encoding

service_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df[service_cols] = np.where(df[service_cols] == 'No internet service', 'No', df[service_cols])
df[service_cols] = df[service_cols].replace({'Yes': 1, 'No': 0}).astype(int)

In [9]:
# Normalize the phone service label and encode the binary field

df['MultipleLines'] = df['MultipleLines'].replace({'No phone service': 'No'})
df['MultipleLines'] = df['MultipleLines'].map({'Yes': 1, 'No': 0})

In [10]:
# Encode contract as an ordinal feature and one-hot encode remaining categorical variables

contract_mapping = {
    'Month-to-month': 0,
    'One year': 1,
    'Two year': 2,
}

df['Contract'] = df['Contract'].map(contract_mapping)

categorical_cols = ['InternetService', 'PaymentMethod']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [11]:
print(df.head(2))

   customerID  gender  SeniorCitizen  Partner  Dependents  tenure  \
0  7590-VHVEG       1              0        1           0       1   
1  5575-GNVDE       0              0        0           0      34   

   PhoneService  MultipleLines  OnlineSecurity  OnlineBackup  ...  Contract  \
0             0              0               0             1  ...         0   
1             1              0               1             0  ...         1   

   PaperlessBilling  MonthlyCharges  TotalCharges  Churn  \
0                 1           29.85         29.85      0   
1                 0           56.95       1889.50      0   

   InternetService_Fiber optic  InternetService_No  \
0                        False               False   
1                        False               False   

   PaymentMethod_Credit card (automatic)  PaymentMethod_Electronic check  \
0                                  False                            True   
1                                  False                  

In [12]:
# Remove the customer ID column and inspect the transformed dataset

df = df.drop(columns=['customerID'])
print(df.head())

   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       1              0        1           0       1             0   
1       0              0        0           0      34             1   
2       0              0        0           0       2             1   
3       0              0        0           0      45             0   
4       1              0        0           0       2             1   

   MultipleLines  OnlineSecurity  OnlineBackup  DeviceProtection  ...  \
0              0               0             1                 0  ...   
1              0               1             0                 1  ...   
2              0               1             1                 0  ...   
3              0               1             0                 1  ...   
4              0               0             0                 0  ...   

   Contract  PaperlessBilling  MonthlyCharges  TotalCharges  Churn  \
0         0                 1           29.85         29.85     

In [13]:
# Check feature relationships to understand multicollinearity and feature importance

corr_matrix = df.corr(numeric_only=True)
print(corr_matrix)

                                         gender  SeniorCitizen   Partner  \
gender                                 1.000000       0.001874  0.001808   
SeniorCitizen                          0.001874       1.000000  0.016479   
Partner                                0.001808       0.016479  1.000000   
Dependents                            -0.010517      -0.211185  0.452676   
tenure                                -0.005106       0.016567  0.379697   
PhoneService                           0.006488       0.008576  0.017706   
MultipleLines                          0.008414       0.142948  0.142057   
OnlineSecurity                         0.017021      -0.038653  0.143106   
OnlineBackup                           0.013773       0.066572  0.141498   
DeviceProtection                       0.002105       0.059428  0.153786   
TechSupport                            0.009212      -0.060625  0.119999   
StreamingTV                            0.008393       0.105378  0.124666   
StreamingMov

### Correlation check

After the encoding fixes above, no pair of features is perfectly correlated. Main observations:

- **Strongest relationships with churn:** `Contract` (-0.40), `tenure` (-0.35), `InternetService_Fiber optic` (+0.31) and `PaymentMethod_Electronic check` (+0.30). Customers on longer contracts and with longer tenure churn less; fiber and electronic-check customers churn more.
- **`gender` is essentially unrelated to churn** (0.009).
- **Strong feature-to-feature correlations remain:** `tenure` and `TotalCharges` (0.83), `MonthlyCharges` and `InternetService_Fiber optic` (0.79). These are expected (longer tenure means more total billed) and are not duplicates, but they can make individual coefficients less stable.

In [14]:
# Prepare the target and feature matrix

X = df.drop(columns=['Churn'])
y = df['Churn']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

X shape: (7043, 22)
y shape: (7043,)


## 3. Baseline models: does scaling matter?

The data is split 80/20 (`random_state=42`) into training and test sets. A logistic regression is trained on raw features and on standardized features (the scaler is fitted on the training set only, then applied to the test set, to avoid leaking test information).

Accuracy alone is misleading here: roughly 73.5% of customers do not churn, so a model that always predicts "stays" would already score about 73.5%.

In [15]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [16]:
# Train the baseline model without scaling

base_model = LogisticRegression(max_iter=10000)
base_model.fit(X_train, y_train)
predictions_base = base_model.predict(X_test)

In [17]:
# Standardize the feature values to improve optimization stability

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
# Fit the model on scaled features and generate predictions

scaled_model = LogisticRegression(max_iter=10000)
scaled_model.fit(X_train_scaled, y_train)
predictions_scaled = scaled_model.predict(X_test_scaled)

In [19]:
# Calculate evaluation metrics for both versions of the model

acc_base = accuracy_score(y_test, predictions_base)
prec_base = precision_score(y_test, predictions_base)
recall_base = recall_score(y_test, predictions_base)
f1_base = f1_score(y_test, predictions_base)
confusion_base = confusion_matrix(y_test, predictions_base)

acc_scaled = accuracy_score(y_test, predictions_scaled)
prec_scaled = precision_score(y_test, predictions_scaled)
recall_scaled = recall_score(y_test, predictions_scaled)
f1_scaled = f1_score(y_test, predictions_scaled)
confusion_scaled = confusion_matrix(y_test, predictions_scaled)

In [20]:
print('Unscaled model:')
print(f'Accuracy: {(acc_base * 100):.2f}%')
print(f'Precision: {(prec_base * 100):.2f}%')
print(f'Recall: {(recall_base * 100):.2f}%')
print(f'F1 score: {(f1_base * 100):.2f}%')

print('\nScaled model:')
print(f'Accuracy: {(acc_scaled * 100):.2f}%')
print(f'Precision: {(prec_scaled * 100):.2f}%')
print(f'Recall: {(recall_scaled * 100):.2f}%')
print(f'F1 score: {(f1_scaled * 100):.2f}%')

Unscaled model:
Accuracy: 82.19%
Precision: 68.60%
Recall: 60.32%
F1 score: 64.19%

Scaled model:
Accuracy: 81.97%
Precision: 68.20%
Recall: 59.79%
F1 score: 63.71%


In [21]:
print('Confusion matrix for unscaled model:\n', confusion_base)
print('\nConfusion matrix for scaled model:\n', confusion_scaled)

Confusion matrix for unscaled model:
 [[933 103]
 [148 225]]

Confusion matrix for scaled model:
 [[932 104]
 [150 223]]


### Baseline results

| Model | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| Unscaled | 82.19% | 68.60% | 60.32% | 64.19% |
| Scaled | 82.04% | 68.52% | 59.52% | 63.70% |

- **Scaling barely changed the result.** `LogisticRegression` uses the `lbfgs` solver with `max_iter=10000`, which converges on raw features too. The small difference comes from the default L2 penalty, which depends on feature scale. Scaling mattered a lot for the from-scratch gradient descent in the breast cancer project, but matters little here.
- **Recall is the weak point.** At the default 0.5 threshold the scaled model misses 151 of the 373 churners in the test set (recall 59.5%). For the stated aim, that is too many missed customers, which motivates the next section.

## 4. Handling class imbalance: threshold tuning and class weighting

Two levers are tested to raise recall on the churn class:

1. **Lower the decision threshold** below 0.5, so customers are flagged as churners at a lower predicted probability.
2. **Class weighting** (`class_weight='balanced'`), which penalizes missing a churner more heavily during training.

Both are evaluated at thresholds 0.5, 0.4, 0.3 and 0.2. Each one trades precision for recall, so the goal is to find a point where recall is high without flooding the retention team with false alarms.

In [22]:
# Test multiple thresholds to evaluate the trade-off between recall and precision

probs = scaled_model.predict_proba(X_test_scaled)[:, 1]
thresholds = [0.5, 0.4, 0.3, 0.2]

for threshold in thresholds:
    preds = (probs >= threshold).astype(int)
    print(f'At threshold {threshold}:')
    print(f'Precision: {precision_score(y_test, preds):.4f}')
    print(f'Recall: {recall_score(y_test, preds):.4f}')
    print(f'Confusion matrix:\n{confusion_matrix(y_test, preds)}')
    print('-' * 50)

At threshold 0.5:
Precision: 0.6820
Recall: 0.5979
Confusion matrix:
[[932 104]
 [150 223]]
--------------------------------------------------
At threshold 0.4:
Precision: 0.6042
Recall: 0.6997
Confusion matrix:
[[865 171]
 [112 261]]
--------------------------------------------------
At threshold 0.3:
Precision: 0.5568
Recall: 0.8016
Confusion matrix:
[[798 238]
 [ 74 299]]
--------------------------------------------------
At threshold 0.2:
Precision: 0.4684
Recall: 0.8740
Confusion matrix:
[[666 370]
 [ 47 326]]
--------------------------------------------------


In [23]:
# Train a balanced logistic regression model to handle class imbalance

balanced_model = LogisticRegression(class_weight='balanced', max_iter=1000)
balanced_model.fit(X_train_scaled, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to

In [24]:
# Evaluate the balanced model across several threshold levels

balanced_probs = balanced_model.predict_proba(X_test_scaled)[:, 1]

for threshold in thresholds:
    preds = (balanced_probs >= threshold).astype(int)
    print(f'Balanced model at threshold {threshold}:')
    print(f'Precision: {precision_score(y_test, preds):.4f}')
    print(f'Recall: {recall_score(y_test, preds):.4f}')
    print(f'Confusion matrix:\n{confusion_matrix(y_test, preds)}')
    print('-' * 50)

Balanced model at threshold 0.5:
Precision: 0.5168
Recall: 0.8231
Confusion matrix:
[[749 287]
 [ 66 307]]
--------------------------------------------------
Balanced model at threshold 0.4:
Precision: 0.4622
Recall: 0.8847
Confusion matrix:
[[652 384]
 [ 43 330]]
--------------------------------------------------
Balanced model at threshold 0.3:
Precision: 0.4298
Recall: 0.9437
Confusion matrix:
[[569 467]
 [ 21 352]]
--------------------------------------------------
Balanced model at threshold 0.2:
Precision: 0.3882
Recall: 0.9732
Confusion matrix:
[[464 572]
 [ 10 363]]
--------------------------------------------------


### Reading the threshold and class-weight results

| Model | Threshold | Precision | Recall | FN | FP |
|---|---|---|---|---|---|
| Unweighted | 0.5 | 68.5% | 59.5% | 151 | 102 |
| Unweighted | 0.3 | 55.4% | 79.9% | 75 | 240 |
| Unweighted | 0.2 | 47.0% | 87.4% | 47 | 368 |
| Balanced | 0.5 | 51.3% | 82.3% | 66 | 291 |
| Balanced | 0.3 | 42.8% | 94.4% | 21 | 470 |
| Balanced | 0.2 | 38.5% | 97.3% | 10 | 580 |

- Both levers move the model along **roughly the same precision/recall trade-off**. Class weighting at 0.5 (recall 82.3%) lands between the unweighted model at thresholds 0.3 and 0.2, so the two approaches are close to interchangeable for logistic regression.
- Pushing recall as high as possible is not automatically best. At threshold 0.2 with class weighting, only 10 churners are missed, but 580 of the 1,036 loyal customers (56%) would receive a retention offer.

## 5. Final model

**Selected configuration:** logistic regression on standardized features with `class_weight='balanced'` and the default threshold of 0.5.

**Why this one:** it catches over 4 in 5 churners (recall 82.3%) while keeping false alarms to 28% of loyal customers. More aggressive settings gain a few points of recall at a steep precision cost (see the table above). The unweighted model at threshold 0.3 is a close alternative (recall 79.9%, 51 fewer false positives); the balanced model was chosen because it puts slightly more weight on catching churners, which matches the aim.

In [26]:
# Side-by-side comparison of the key configurations on the test set
configs = {
    'Baseline (scaled, threshold 0.5)':         (probs, 0.5),
    'Unweighted, threshold 0.3':                (probs, 0.3),
    'Balanced, threshold 0.5  <-- FINAL':       (balanced_probs, 0.5),
    'Balanced, threshold 0.2':                  (balanced_probs, 0.2),
}

rows = []
for name, (p, t) in configs.items():
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({
        'Configuration': name,
        'Accuracy %': accuracy_score(y_test, pred) * 100,
        'Precision %': precision_score(y_test, pred) * 100,
        'Recall %': recall_score(y_test, pred) * 100,
        'F1 %': f1_score(y_test, pred) * 100,
        'FN': fn,
        'FP': fp,
    })

summary = pd.DataFrame(rows).set_index('Configuration')
print(summary.round(1).to_string())

                                    Accuracy %  Precision %  Recall %  F1 %   FN   FP
Configuration                                                                        
Baseline (scaled, threshold 0.5)          82.0         68.2      59.8  63.7  150  104
Unweighted, threshold 0.3                 77.9         55.7      80.2  65.7   74  238
Balanced, threshold 0.5  <-- FINAL        74.9         51.7      82.3  63.5   66  287
Balanced, threshold 0.2                   58.7         38.8      97.3  55.5   10  572


In [27]:
# Final model: class-weighted logistic regression at threshold 0.5
final_threshold = 0.5
final_preds = (balanced_probs >= final_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, final_preds).ravel()
final_precision = precision_score(y_test, final_preds)
final_recall = recall_score(y_test, final_preds)
base_churn_rate = y_test.mean()

print(f'FINAL MODEL (class_weight=balanced, threshold={final_threshold})')
print('Confusion matrix:')
print(confusion_matrix(y_test, final_preds))
print(f'Precision: {final_precision:.2%}')
print(f'Recall:    {final_recall:.2%}')
print(f'F1 score:  {f1_score(y_test, final_preds):.2%}')
print(f'Accuracy:  {accuracy_score(y_test, final_preds):.2%}')
print(f'\nChurners in test set: {tp + fn} | caught: {tp} | missed (FN): {fn}')
print(f'Customers flagged: {tp + fp} of {len(y_test)} ({(tp + fp) / len(y_test):.1%}) | false alarms (FP): {fp}')
print(f'Base churn rate: {base_churn_rate:.1%} | precision lift over random targeting: {final_precision / base_churn_rate:.2f}x')

FINAL MODEL (class_weight=balanced, threshold=0.5)
Confusion matrix:
[[749 287]
 [ 66 307]]
Precision: 51.68%
Recall:    82.31%
F1 score:  63.50%
Accuracy:  74.95%

Churners in test set: 373 | caught: 307 | missed (FN): 66
Customers flagged: 594 of 1409 (42.2%) | false alarms (FP): 287
Base churn rate: 26.5% | precision lift over random targeting: 1.95x


## 6. Conclusion

**Final result (held-out test set, 1,409 customers, 373 of whom churned):** logistic regression with standardized features and `class_weight='balanced'` at threshold 0.5 achieves **precision 51%, recall 82%, 66 false negatives and 291 false positives** (307 churners caught, 745 loyal customers correctly left alone).

**What this means in practice**
- The model identifies 307 of the 373 customers who left. Only 66 are missed, down from 151 for the baseline model (a 56% reduction).
- It flags 598 customers (42% of the test set) for a retention offer. About half of them (51%) really would churn, versus a base churn rate of 26.5%, so targeting the flagged group is roughly 1.9 times more efficient than contacting customers at random.
- The cost is 291 loyal customers who would receive an offer they did not need. Under the assumption that a missed churner costs more than an unneeded offer, this is a deliberate trade.

**Why accuracy was not used to choose the model:** accuracy fell from 82.0% (baseline) to 74.7% (final), which is barely above the 73.5% of a model that always predicts "stays". The baseline looked better on accuracy only because it ignored most churners.

**Key findings from the data**
- Contract type and tenure are the strongest predictors: month-to-month customers and new customers churn far more.
- Fiber optic internet and electronic-check payment are associated with higher churn; gender shows no meaningful relationship.
- One-hot encoding the add-on service columns created perfectly redundant dummy variables; collapsing "No internet service" into "No" removed the problem.
- Feature scaling had almost no effect on `LogisticRegression` here, unlike in the from-scratch implementation.
- Class weighting and threshold tuning both trade precision for recall along a similar curve.

**Limitations**
- The threshold and class-weight settings were compared on the same test set that produces the final numbers, so the reported figures are slightly optimistic. A cleaner design would choose them on a validation set or with cross-validation and use the test set only once.
- Results come from a single 80/20 split without stratification; with 373 churners in the test set, metrics will shift somewhat across splits.
- Recall was prioritized by reasoning, not from real cost figures. With actual retention-offer cost and customer lifetime value, the threshold could be set by expected profit instead.
- The 11 missing `TotalCharges` values were filled with the median, which may not be the most appropriate value for brand-new customers.

**Possible next steps:** use stratified k-fold cross-validation to choose the threshold, add a cost-based threshold, compare against non-linear models (e.g. decision trees or neural networks) after the next courses, and wrap the model in a small Streamlit app that returns a churn probability for a single customer.